# Davinci Notebook 13 — Agent & Tool-Use Training Data

Build a validated JSONL schema for agent trajectories: planning (optional), tool selection, JSON arguments, observations, final response, and error recovery. It does not synthesize verified trajectories; validate examples and tool outcomes independently.

In [ ]:
import json, hashlib
from pathlib import Path


## 1. Canonical trajectory schema
Example format includes explicit tool result provenance and avoids treating observation text as model instructions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import sys, subprocess, base64
REPO_DIR=Path("/content/Davinci")
if not (REPO_DIR/".git").exists():
    from google.colab import userdata
    token=userdata.get("GITHUB_TOKEN")
    if not token: raise RuntimeError("Add GitHub Secret GITHUB_TOKEN with Contents: read permission.")
    encoded=base64.b64encode(("x-access-token:"+token).encode()).decode()
    subprocess.run(["git","-c",f"http.extraheader=AUTHORIZATION: basic {encoded}","clone","--depth","1","https://github.com/keyhmoham/Davinci.git",str(REPO_DIR)],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci.trajectory import audit_trajectories, export_jsonl, split_trajectories, trajectory_hash, validate_trajectory
REPO_COMMIT=subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip()
print("Trajectory validator revision:",REPO_COMMIT)

In [ ]:
example={
 "trajectory_id":"example-001","language":"fa",
 "messages":[
  {"role":"user","content":"What is 19 + 23?"},
  {"role":"assistant","type":"tool_call","name":"add","arguments":{"left":19,"right":23}},
  {"role":"tool","name":"add","content":"{\"sum\":42}","trusted":False},
  {"role":"assistant","content":"42"},
 ],
 "metadata":{"source":"synthetic","verified":False},
}
print("Schema example issues:",validate_trajectory(example))
print("Example hash:",trajectory_hash(example))

## 2. Dataset quality and duplicate detection
Use a stable hash to detect exact duplicates; this is not semantic deduplication.

In [ ]:
TRACE_JSONL_PATH=Path("/content/drive/MyDrive/Davinci/sft_data/processed/agent_tool_traces.jsonl")
OUTPUT_DIR=Path("/content/drive/MyDrive/Davinci/agent_trajectories/processed")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if not TRACE_JSONL_PATH.is_file():
    raise FileNotFoundError(f"Recorded trajectory JSONL not found: {TRACE_JSONL_PATH}")

trajectories=[]; malformed=[]
with TRACE_JSONL_PATH.open(encoding="utf-8") as handle:
    for line_number,line in enumerate(handle,1):
        if not line.strip(): continue
        try: trajectories.append(json.loads(line))
        except json.JSONDecodeError as exc: malformed.append({"line":line_number,"error":str(exc)})

report=audit_trajectories(trajectories)
report["malformed_json_lines"]=malformed
valid=[item for item in trajectories if not validate_trajectory(item)]
if not valid: raise ValueError("No valid, complete tool trajectories were found.")
train_trajectories,val_trajectories=split_trajectories(valid,validation_fraction=0.05,seed=42)
print(json.dumps({k:v for k,v in report.items() if k not in {"issues","duplicate_rows"}},ensure_ascii=False,indent=2))
print("Split sizes:",len(train_trajectories),len(val_trajectories))

## 3. Export JSONL and dataset card fields

In [ ]:
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()

train_path=OUTPUT_DIR/"agent_train.jsonl"
validation_path=OUTPUT_DIR/"agent_validation.jsonl"
train_written=export_jsonl(train_trajectories,train_path)
validation_written=export_jsonl(val_trajectories,validation_path)
audit_path=OUTPUT_DIR/"audit_report.json"
audit_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
dataset_card={
 "sources":"Fill with source datasets and/or runtime provenance",
 "collection_method":"Recorded tool trajectories; no synthetic success outcomes added by this notebook",
 "license":"Review every source and upstream license before training or redistribution",
 "verification_method":"Strict schema validation and tool-call/result pairing; task correctness requires separate review",
 "tool_environment":"Record tool versions, permissions, and execution environment",
 "privacy_review":"Pending",
 "known_limitations":["Exact duplicates only","No semantic deduplication","No automatic correctness verification"],
 "version":"0.1.0",
}
card_path=OUTPUT_DIR/"DATASET_CARD_TEMPLATE.json"
card_path.write_text(json.dumps(dataset_card,ensure_ascii=False,indent=2),encoding="utf-8")
manifest={
 "repository_commit":REPO_COMMIT,
 "source_file":str(TRACE_JSONL_PATH),
 "source_sha256":sha256_file(TRACE_JSONL_PATH),
 "source_trajectories":len(trajectories),
 "valid_trajectories":len(valid),
 "invalid_trajectories":report["invalid"],
 "exact_duplicates":report["duplicates"],
 "malformed_json_lines":len(malformed),
 "train_written":train_written,
 "validation_written":validation_written,
 "validation_fraction":0.05,
 "seed":42,
 "limitations":["No near-duplicate detection","No tool-schema manifest validation","No task correctness proof"],
}
(OUTPUT_DIR/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print("Wrote:",train_path,validation_path,audit_path,card_path,sep="\n")

## 4. Trajectory types to collect
- Tool choice and tool argument generation.
- Multi-tool sequencing and stopping decisions.
- Error handling and retry policies.
- Replanning after failed/partial observations.
- Tool-free tasks (recognizing when tools are unnecessary).
- Persian-language tools and bilingual interactions.
- MCP tools with permission boundaries and untrusted outputs.
- Browser, code, file, database, calculator, and retrieval tasks.
- Long task history with relevant-memory retrieval.

Use a mix of human-authored, verified programmatic, and carefully audited synthetic examples; record provenance and license.